# Week 6 (Tree-based Models)

## 2.1. Decision Tree

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import graphviz
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn import datasets
from sklearn.model_selection import train_test_split

In [ ]:
# 데이터 로드 및 전처리
iris = datasets.load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

# 훈련/테스트 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Q1.1 max_depth 파라미터를 변경하면 트리의 구조와 성능이 어떻게 변할까요?

max_depth를 늘리면 트리가 더 세분화된 규칙까지 학습하면서 leaf 개수와 노드 수가 늘어나고, training 정확도는 계속 올라간다. 하지만 일정 깊이를 넘어서면 노이즈까지 학습해 overfitting이 발생해 test 성능은 오히려 떨어질 수 있다. 반대로 max_depth를 줄이면 트리가 단순해지고 해석은 쉬워지지만, 패턴을 충분히 학습하지 못해 underfitting이 발생할 수 있다. 따라서 적절한 max_depth는 validation 성능을 기준으로 탐색해야 한다.

In [ ]:
# 결정 트리 모델 생성 및 학습
tree_model = DecisionTreeClassifier(criterion="gini", max_depth=3, min_samples_leaf=2, random_state=42)
tree_model.fit(X_train, y_train)


### Q1.2 'gini'와 'entropy' 기준의 차이는 무엇이며, 결과에 어떤 영향을 미칠까요?

gini는 지니 불순도($1-\sum p_i^2$)를, entropy는 정보 엔트로피($-\sum p_i \log p_i$)를 분할 기준으로 사용한다. 두 지표 모두 노드의 불순도(클래스 혼합 정도)를 측정한다는 점은 동일하지만, entropy는 log 연산이 포함되어 계산 비용이 더 크고 불순도 변화에 조금 더 민감하게 반응하는 경향이 있다. 실무적으로는 두 기준으로 학습한 트리의 구조와 성능 차이가 크지 않은 경우가 대부분이다.

### Q1.3 각 노드의 분할 기준 어떻게 되는지 설명해보세요.

각 노드에서는 가능한 모든 feature와 threshold 조합에 대해 분할 후 자식 노드들의 불순도(gini 또는 entropy)를 계산하고, 분할 전후 불순도 감소량(information gain)이 가장 큰 조합을 선택한다. 이 과정을 각 노드마다 greedy하게 반복하면서(CART 알고리즘) 트리를 재귀적으로 성장시키며, min_samples_leaf 등 정지 조건에 도달하면 분할을 멈춘다.

In [ ]:
# 트리 시각화 (Graphviz)
dot_data = export_graphviz(
    tree_model,
    out_file=None,
    feature_names=X.columns,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    special_characters=True
)

In [ ]:
# 그래프 출력
graph = graphviz.Source(dot_data)
graph

## 3. 앙상블 학습: 부스팅(Boosting) 모델

Adaboost, Gradient Boosting Model은 scikit learn에서 지원하지만, LightGBM과 XGBoost의 경우는 외부 패키지를 불러와야 합니다.

Boosting 계열의 모델들은 데이터 개수가 적으면 overfitting이 일어나는 경우가 많으니, 주의해야 합니다.

In [ ]:
# 필요한 패키지 로드
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error

housing = fetch_california_housing()

In [ ]:
# 데이터 로드
housing_df = pd.DataFrame(housing.data, columns=housing.feature_names)
housing_df['MedHouseVal'] = housing.target

In [ ]:
housing_df.head()

### Q3.1 `housing_df`를 train, test split을 해봅시다.

분할 비율을 자유롭게 설정해봅시다.

In [ ]:
X = housing_df.drop(columns=['MedHouseVal'])
y = housing_df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Q3.2  데이터를 `RandomForestRegressor`에 적합해 봅시다.
train set 에 대해 `MedHouseVal`을 종속변수, 나머지를 독립변수로 하는 random forest regressor를 적합해 봅시다.

test set으로 prediction을 한 후 MSE를 구해 봅시다.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_mse

### Q3.3 데이터를 `AdaBoostRegressor`에 적합해봅시다.

위와 동일

In [ ]:
from sklearn.ensemble import AdaBoostRegressor

ada_model = AdaBoostRegressor(random_state=42)
ada_model.fit(X_train, y_train)
ada_pred = ada_model.predict(X_test)
ada_mse = mean_squared_error(y_test, ada_pred)
ada_mse

### Q3.4 데이터를 `GradientBoostingRegressor`에 적합해 봅시다.

위와 동일

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(random_state=42)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_mse = mean_squared_error(y_test, gb_pred)
gb_mse

### Q3.5 데이터를 `lightgbm` 회귀에 적합해 봅시다.

파라미터를 자세히 살펴보고, 자유롭게 설정해 봅시다.

Documentation
https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMRegressor.html

예시
https://www.geeksforgeeks.org/regression-using-lightgbm/


In [ ]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)
lgb_mse = mean_squared_error(y_test, lgb_pred)
lgb_mse

### Q3.6 데이터를 `xgboost`에 적합해 봅시다.

파라미터는 자유롭게 설정해 봅시다

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_mse

### Q3.7 `RandomForestRegressor`의 feature importance를 시각화해봅시다.

https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html

위를 참고해보면서 Q2에서 적합한 random forest regressor의 feature importance를 시각화해 봅시다.

중요한 feature 부터 내림차순으로 시각화해 봅시다.

In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), X.columns[indices], rotation=45, ha='right')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

### (BONUS) 1. Boosting 모델 각각의 특징을 정리해보세요.

- **AdaBoost**: 이전 모델이 잘못 예측한 샘플의 가중치를 높여가며 약한 학습기(주로 얕은 트리)를 순차적으로 추가. 이상치에 민감함.
- **Gradient Boosting**: 이전 모델의 잔차(gradient)를 학습 대상으로 삼아 순차적으로 트리를 추가. AdaBoost보다 유연하지만 학습이 느리고 과적합 위험이 있음.
- **LightGBM**: leaf-wise 트리 성장과 histogram 기반 분할로 대용량 데이터에서도 학습 속도가 빠름. 데이터가 적으면 과적합되기 쉬움.
- **XGBoost**: level-wise 트리 성장과 L1/L2 정규화를 결합해 과적합을 억제하며, 병렬화와 결측치 자동 처리를 지원함.

### (BONUS) 2. 모델 성능을 높이기 위하여 어떤 시도를 하면 좋을지 고민해보세요.

- 하이퍼파라미터 튜닝(learning_rate, max_depth, n_estimators 등)을 GridSearch나 Optuna로 최적화
- feature engineering으로 유의미한 파생 변수 추가, 불필요한 변수 제거
- cross-validation으로 일반화 성능을 안정적으로 평가
- stacking, voting 등 여러 모델을 결합하는 앙상블 시도
- 이상치·결측치 처리 등 데이터 전처리 품질 개선